# DeepONets and Interest-Rate Curves

It turns out -perhaps not so surprising by this point- that we can train neural networks to take market quotes (deposits, FRAs/futures, swaps, \ldots) and produce a discount curve such that those same quotes are repriced correctly. On paper this sounds straightforward: learn a mapping from inputs to discount factors. In practice, it is a bit more subtle. In this post we will see why it can be an interesting approach, how it can be implemented, and what things must be kept in mind in an approach like this.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd
import plotly

from plots import *
from solvers import *
from utils import *

torch.manual_seed(0)
DEVICE = torch.device('cpu')  # change to 'cuda' if you have a GPU
DTYPE = torch.float64

# Load market data
market_rates_df = pd.read_excel("swaps.xlsx", index_col='date', parse_dates=['date'])
market_rates_df = market_rates_df.sort_index().dropna()
market_rates_df /= 100 

## Lets recap some definitions

Before jumping into the implementation, I think it is worth taking a second to review a bit about interest-rate modeling. Starting with interest-rate swaps, in general their price can be described by the formula

$$
\Pi_t
=
N\left(
\underbrace{\sum_{i=1}^{n}\alpha_i\,P(t,T_i)\,F(t;T_{i-1},T_i)}_{\text{PV floating leg}}
-
\underbrace{K\sum_{i=1}^{n}\alpha_i\,P(t,T_i)}_{\text{PV fixed leg}}
\right),
$$

where $N$ is the notional, $K$ is the fixed rate, $\alpha_i$ is the year fraction (accrual factor) of the $i$-th coupon, $T_i$ are the payment dates, and $P(t,T)$ is the discount factor. Assuming a single-curve framework, the forward rate for the period $[T_{i-1},T_i]$ is the (simple) forward rate $F(t;T_{i-1},T_i)$. If the swap is entered at par, we can impose $\Pi_t=0$ and obtain the par swap rate

$$
S(t)=K^{\star}(t)=
\frac{\sum_{i=1}^{n}\alpha_i\,P(t,T_i)\,L(t;T_{i-1},T_i)}
{\sum_{i=1}^{n}\alpha_i\,P(t,T_i)}.
$$

Once we obtain each discount factor via bootstrapping, we recover forward rates through

$$
F(t;T_{i-1},T_i)
=
\frac{1}{\alpha_i}\left(\frac{P(t,T_{i-1})}{P(t,T_i)}-1\right).
$$

### Curve characteristics

Given these definitions and identities, we can see what mathematical properties we *would like* these components to have:

- Strictly positive discount factors $P(t,T)>0$: otherwise we would be allowing prices that could be absurd, such as negative values for a risk-free instrument, which swaps generally are.
- Reasonable monotonicity: we expect monotonicity in discount curves if we assume positive rates. If we allow negative rates, we still want some degree of monotonicity to avoid oscillations in forward rates.
- Smoothness: this matters because we want to be able to compute **instantaneous forward rates**.

Indeed, once we have $P(t,T)$ for all $T$, we can define the instantaneous forward rate as

$$
f(t,T)=-\frac{\partial}{\partial T}\log P(t,T).
$$

Clearly, the instantaneous forward rate requires $P(t,T)$ to be differentiable in $T$ and smooth enough to avoid “jumps” in forward rates. The idea is to prevent a small kink in $\log P$ from turning into a significant jump in $f$, or a small oscillation from turning into a saw-tooth-shaped forward curve. In practice, unreasonable instantaneous forward curves are not just an aesthetic issue: they can generate unrealistic carry/roll-down profiles and make model calibration harder.

Now, where do discount factors come from in the first place? In continuous time, discounting is linked to the short rate $r(t)$ by

$$
P(t,T)=\mathbb{E}_t^\mathbb{Q}\left[\exp\left(-\int_t^T r_u\,du\right)\right]
=
\mathbb{E}_t^\mathbb{Q}\left[\exp\left(-\int_t^T f(t,u)\,du\right)\right].
$$

At time $t=0$ for curve construction, we often treat $P(0,T)$ as the primitive object that we infer from market quotes. Once we have $P(0,T)$, we can obtain zero rates $z(0,T)$ via $P(0,T)=e^{-z(0,T)T}$ (in the continuously compounded case), and then instantaneous forwards using the derivative above. Given this, the big question is: how do we obtain these instantaneous forward rates if we only have *discrete data points*?

In practice, we only observe a finite set of instruments: deposits, futures/FRAs, swaps at a handful of maturities. These give us the pillars $\{T_1,\dots,T_m\}$ and values $\{P(0,T_j)\}$ (or equivalently zero rates, par rates, etc.). However, between pillars there are infinitely many curves that reproduce the same market prices, so we must *choose* an interpolation rule (and sometimes an extrapolation rule) in order to have a description of what happens inside each segment.

### Curves are determined by assumptions

If we interpolate discount factors, we can preserve positivity naturally, especially if we interpolate $\log P$ (piecewise-linear interpolation in $\log P$ keeps $P>0$ by construction). If we interpolate zero rates, we often obtain a visually smooth curve, but we may inadvertently introduce non-local effects in the forwards. If we interpolate directly on instantaneous forwards, we can control smoothness very well, but we are making a strong structural assumption (we are literally specifying the object that we then “make exist” by integration).

In the real world, there is no free lunch: local, shape-preserving interpolations (piecewise linear, monotone convex) are robust and tend not to generate oscillations, but they can create kinks (lack of smoothness of $f$) that show up as jumps in forwards. Global, smooth interpolations (cubic splines) look very nice in zero rates, but they can overshoot between sparse pillars and create negative “humps” or ripples in $f$. In other words, interpolation is not a cosmetic detail: it is a modeling choice about what is happening in the market between the quotes we actually observe.

Something that is not often understood is that, because we have only a finite number of prices, there are *infinitely* many instantaneous forward curves consistent with the observed prices. For example, we could shift $f(0,T)$ up over one interval and down over another and still preserve the same integrals that determine the discount factors at the pillars. In a sense, the market helps us pin down the averages (expectations under $\mathbb{Q}$), and the curve builder's job is to choose a particular representation of the continuous phenomenon.


In [2]:
plot_interpolation_examples(market_rates_df, market_rates_df.index[100])

### A data-driven process

Given the above, one might think that a data-driven approach is naturally superior to one where the user must impose structure to achieve uniqueness of the instantaneous forward curve; however, in this context neural networks have the same problem: from a finite set of quotes, there exist infinitely many compatible curves, and the model must choose one. The difference is that, instead of explicitly choosing an interpolation rule, we choose a *function class* (the architecture), a *fitting criterion* (a *loss function*), and some form of *regularization*.

The question, then, is not whether a neural network eliminates modeling decisions, but *where* we are placing them and *how* we control them. The key is that we can learn these conditions from market observations: rather than hand-coding *how the curve should look* between pillars, we train a model to learn an *inductive bias* that reflects historical patterns (by currency, rate regime, liquidity, etc.). In practice, this translates into several potential advantages:

- **Speed and computational scale**: Once trained, the network produces a curve (or its parameters) in a single evaluation. This is useful when curves must be built thousands of times (XVA, simulation, intraday) or when multiple curves are needed at once.

- **Handling incomplete or noisy data**: In real data it is common to have missing maturities, clustered quotes, or *outliers*. Instead of relying on *ad hoc* heuristics (discarding, filling, manual smoothing), the model can be trained to be robust to these issues (e.g., via instrument *dropout* and losses that are less sensitive to *outliers*).

- **Product-specific characteristics**: Deposits, FRAs/futures, and swaps “measure” and behave differently; likewise, different currencies may exhibit different curvature in the short-to-medium segment. In an NN pipeline, repricing can be implemented with differentiable product-specific layers, and the model can learn effective weights that reflect liquidity, *bid--ask*, and segment relevance.

- **Temporal coherence and consistency with the forward model**: Stochastic interest-rate models typically take an initial instantaneous forward curve as input. For this reason, it is convenient to enforce that the constructed curve is “compatible” with the chosen dynamics, so that on day zero we start from a feasible state within the model itself. This consistency also reduces mismatches between “curve” and “model” and helps avoid constant recalibration of the parameters that govern the stochastic dynamics.

- **Automatic differentiation**: Since everything is differentiable, we can compute sensitivities of outputs with respect to inputs instantly via automatic differentiation.

In short, using an NN does not remove the non-uniqueness problem; what it does is turn it into an explicit choice of *prior* (architecture + regularization) that can be learned from data. The practical promise is not to obtain a “magically true” curve, but rather a fast, robust, and coherent builder that produces reasonable forwards without so much forced logic.


## Experiments

Now let's move on to some implementations. For the goal of learning a bootstrapping function, I've chosen a particular architecture: ***DeepONets***, which are designed to learn mappings from functions to functions. All the code is available on my [GitHub page](https://github.com/jmelo11/).

#### Aside: Training setup and data

The data used were obtained from the Central Bank of Chile's website and include IR swap curves from 2008 to 2025. As with other machine learning applications, we split the dataset into training and evaluation sets. Note that here we do not care if there is spillover of information between days, since our goal is to train the model independently of the current-time variable $t$. In fact, we want to sample across different times so that our model learns plausible curve shapes from historical data.

In [3]:
# split into train and test
train_size = int(0.8 * len(market_rates_df))
train_market_rates_df = market_rates_df.sample(train_size, random_state=0)
test_market_rates_df = market_rates_df.drop(train_market_rates_df.index)

In [4]:
fig = go.Figure()
sample = market_rates_df.sample(3, random_state=1)
for date, row in sample.iterrows():
    tenors_years = np.array([str_tenor_to_days(c) for c in row.index]) / 360
    fig.add_trace(go.Scatter(x=tenors_years, y=row.values, mode='lines+markers', name=str(date.date())))

fig.update_xaxes(ticktext=market_rates_df.columns, tickvals=tenors_years)
fig.update_yaxes(tickformat=".2%")
fig.update_layout()
apply_style(fig, title="Market Swap Rates - Camara/Fix", x_title="Maturity (years)", y_title="Swap Rate")
fig.show()

### Today's Choice: DeepONets

Training DeepONets can be viewed as learning an *operator* (a mapping from functions to functions) of the form
$$
\mathcal{G}:\; u(\cdot)\ \mapsto\ v(\cdot),
$$
where both the input and the output are functions. In our setting, the input is a *discretely observed* swap curve (pillar quotes across tenors), and the output is a *continuous* discount curve,
$$
T \mapsto P(0,T; t).
$$
Here we treat $t$ as a fixed parameter that simply indicates the evaluation date associated with the input quotes.

A standard DeepONet splits the computation into two networks:

- A **branch** network that reads the input quotes (the vector of pillar swap rates) and produces a latent representation $b \in \mathbb{R}^p$.
- A **trunk** network that reads the query maturity $T$ and produces another representation $h(T)\in\mathbb{R}^p$.

They are combined via an inner product,
$$
g(T) = \langle b,\; h(T)\rangle + \beta,
$$
so the model effectively learns a low-rank representation of the operator. In the implementation below we interpret the output as a continuously compounded zero rate $k(T)$ by applying a sigmoid transformation (optionally with scaling),
$$
k(T)=k_{\min}+(k_{\max}-k_{\min})\,\sigma(g(T)),
\qquad
\sigma(x)=\frac{1}{1+e^{-x}},
$$
and we return the discount factor as
$$
P(0,T; t)=\exp\!\big(-T\,k(T)\big).
$$

This construction ensures $P(0,T; t)>0$ by definition. Monotonicity is not guaranteed automatically (since $k(T)$ is a learned function of $T$), so we control shape indirectly through regularization (e.g., smoothness/convexity penalties and long-end constraints).


In [5]:
class DeepONetBootstrapper(nn.Module, CurveModel):    
    '''
    DeepONet-based bootstrapper for discount curve modeling.
    Uses a branch network to process market rates and a trunk network to process maturities.
    Note: CurveModel is an abstract base class defining the interface for curve models.
    '''
    def __init__(self, n_tenors: int, p: int = 16, hidden: int = 16):
        super().__init__()
        self.n_tenors = n_tenors
        
        self.branch = nn.Sequential(
            nn.Linear(1 * n_tenors, hidden, dtype=DTYPE, device=DEVICE),
            nn.Tanh(),
            nn.Linear(hidden, hidden, dtype=DTYPE, device=DEVICE),
            nn.Tanh(),
            nn.Linear(hidden, p, dtype=DTYPE, device=DEVICE),
        )
        self.trunk = nn.Sequential(
            nn.Linear(1, hidden, dtype=DTYPE, device=DEVICE),
            nn.Tanh(),
            nn.Linear(hidden, hidden, dtype=DTYPE, device=DEVICE),
            nn.Tanh(),
            nn.Linear(hidden, p, dtype=DTYPE, device=DEVICE),
        )
        self.bias = nn.Parameter(torch.zeros(1, dtype=DTYPE, device=DEVICE))
        self._b = None

    def set_curve(self, rates: torch.Tensor, tenors: torch.Tensor, **kwargs) -> None: 
        # in general we would use only rates, but keep tenors for interface consistency       
        if rates.ndim != 1 or tenors.ndim != 1:
            raise ValueError("rates and tenors must be 1D")
        if rates.numel() != self.n_tenors or tenors.numel() != self.n_tenors:
            raise ValueError(f"Expected n_tenors={self.n_tenors}")

        # x = torch.cat([rates], dim=0).unsqueeze(0)  
        self._b = self.branch(rates)       

    def discounts(self, t: torch.Tensor) -> torch.Tensor:
        if self._b is None:
            raise RuntimeError("Call set_curve(rates, tenors) before pricing.")

        t = t.reshape(-1)
        h = self.trunk(t.unsqueeze(-1))                               
        g = (h * self._b).sum(dim=-1) + self.bias                     
        k = F.sigmoid(g)                # if k is like the integral of the fwd rate, sigmoid keeps it bounded
        df = torch.exp(-(t * k)).unsqueeze(-1) # this looks like exp(-t * rate)
        return df
    
model = DeepONetBootstrapper(n_tenors=market_rates_df.shape[1], p=32, hidden=32)
model.apply(weight_init_xavier)

DeepONetBootstrapper(
  (branch): Sequential(
    (0): Linear(in_features=9, out_features=32, bias=True)
    (1): Tanh()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): Tanh()
    (4): Linear(in_features=32, out_features=32, bias=True)
  )
  (trunk): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): Tanh()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): Tanh()
    (4): Linear(in_features=32, out_features=32, bias=True)
  )
)

In [6]:
swaps = [Swap(c, dtype=DTYPE, device=DEVICE) for c in market_rates_df.columns] # utility function, takes a model and computes the par rate of a given maturity.
tenors = torch.tensor([str_tenor_to_days(c) for c in market_rates_df.columns], device=DEVICE, dtype=DTYPE)/360
solver = SSBroyden(model.parameters(), max_iter=2000, tolerance_change=1e-15, tolerance_grad=1e-15)
training_set = gen_training_set(train_market_rates_df, n_pairs=200, device=DEVICE, dtype=DTYPE)

## Loss Functions

For this particular case, we will focus on the following tasks:

- Be able to reproduce swap par rates with a minimum degree of accuracy.
- Be able obtain meaningful sensitivities (with AD).
- Obtain a low-frequency representative from the infinite-dimensional family of instantaneous forward rate curves that match the same pillar prices.

In order to achieve these goals, we train the neural network on loss functions that guide it to map swap quotes into discount factors while controlling what happens between (and beyond) the observed tenors.

Throughout, let $t\in\{1,\dots,m\}$ index the observation dates (days) and $T_i,\ i\in\{1,\dots,n\}$ the swap tenors. The network produces a discount curve $P_\theta(0,T; t)$ for each date $t$, from which we compute the model-implied par rates $\widehat{S}_\theta(T_i; t)$.

### Swap Rate Loss

The simplest, but most important, loss function is the swap rate loss. It measures the deviation from observed par swap rates:

$$
\mathcal{L}_{\text{swap}}(\theta)
=
\frac{1}{mn}\sum_{t=1}^{m}\sum_{i=1}^{n}
\left(\widehat{S}_\theta(T_i; t)-S^{\text{mkt}}(T_i; t)\right)^2,
$$

where $S^{\text{mkt}}(T_i; t)$ denotes the market par rate at tenor $T_i$ on date $t$.

### Long-Term Forwards Loss

Since we only observe instruments up to a final maturity $T_{\max}$, extrapolation beyond $T_{\max}$ requires an additional assumption. Here we impose a ``flat-forward tail'' by penalizing variation of the instantaneous forward curve after $T_{\max}$.

Define the instantaneous forward rate for each date $t$ as

$$
f_\theta(T; t)=-\frac{\partial}{\partial T}\log P_\theta(0,T; t).
$$

Let $T_{\text{tail}}>0$ be the extrapolation horizon and define the average tail forward

$$
\bar f_\theta(t)
=
\frac{1}{T_{\text{tail}}}\int_{T_{\max}}^{T_{\max}+T_{\text{tail}}} f_\theta(u; t)\,du.
$$

Then the long-end loss is

$$
\mathcal{L}_{\text{tail}}(\theta)
=
\frac{1}{m\,T_{\text{tail}}}\sum_{t=1}^{m}
\int_{T_{\max}}^{T_{\max}+T_{\text{tail}}}
\left(f_\theta(T; t)-\bar f_\theta(t)\right)^2\,dT.
$$

This encourages the instantaneous forward rate to be approximately constant over the extrapolation region. If desired, one can replace $\bar f_\theta(t)$ by a fixed ultimate forward rate $f_\infty$ and penalize $\left(f_\theta(T; t)-f_\infty\right)^2$ instead.

### *Spikiness* Loss

A naive smoothness penalty such as $\int (f_\theta'(T;t))^2\,dT$ would suppresses all the curvature and over-flatten the forward curve so we need to come up with another solution. Our goal is more specific: discourage *localized spikes* and high-frequency oscillations while allowing broad, low-frequency curvature. To capture spikiness, we penalize *excess* curvature of the forward curve, using the second derivative $f_\theta(T;t)$ and a tolerance threshold $\kappa>0$. Let

$$
x_\theta(T;t)=\left(\left|\frac{\partial^2}{\partial T^2}f_\theta(T; t)\right|-\kappa\right)_+,
\qquad (y)_+=\max(y,0).
$$

We then define

$$
\mathcal{L}_{\text{spike}}(\theta)
=
\frac{1}{m\,(T_{\max}-T_{\min})}\sum_{t=1}^{m}
\int_{T_{\min}}^{T_{\max}}
\phi\!\left(x_\theta(T;t)\right)\,dT,
$$

where $\phi$ is a nonnegative penalty function. A possible choice could be, for example, the Huber penalty

$$
\phi(x)=
\begin{cases}
\frac{1}{2}x^2, & 0\le x\le \delta,\\[4pt]
\delta\left(x-\frac{1}{2}\delta\right), & x>\delta,
\end{cases}
$$

with $\delta>0$ controlling the transition between quadratic and linear growth. When $\left|f_\theta''(T;t)\right|\le \kappa$, we have $x_\theta(T;t)=0$ and the penalty vanishes, so low-frequency curvature is largely preserved; sharp kinks and oscillations (large $\left|f_\theta''\right|$) are penalized.

### Final Objective

The final loss is given by a weighted combination of the above:

$$
\mathcal{L}(\theta)
=
\mathcal{L}_{\text{swap}}(\theta)
+
\lambda_{\text{tail}}\mathcal{L}_{\text{tail}}(\theta)
+
\lambda_{\text{spike}}\mathcal{L}_{\text{spike}}(\theta),
$$

where $\lambda_{\text{tail}}\ge 0$ and $\lambda_{\text{spike}}\ge 0$ control the trade-off between market fit, tail behaviour, and suppression of forward spikes.


In [ ]:
def closure(*args, **kwargs) -> torch.Tensor:
    solver.zero_grad()
    n = len(training_set.rates_next)    
    sl,le,spl = 0.0,0.0,0.0
    for rates_next in training_set.rates_next:
        sl += swap_rate_loss(model, swaps, rates_next, tenors)*100        
        le += long_end_loss(model, rates_next, tenors, t_last=10.0, mode='ufr', ufr=0.03)
        spl += spike_loss(model, rates_next, tenors)

        if torch.any(torch.isnan(sl)) or torch.any(torch.isnan(le)) or torch.any(torch.isnan(spl)):
            raise ValueError("NaN encountered in loss computation.")
        
    sl = (sl / n).sum()
    le = (le / n).sum()
    spl = (spl / n).sum()
    total_loss = (sl + le + spl)
    total_loss.backward()
    print(f"Current loss: {total_loss.item():.6e}, Swap Loss: {sl.item():.6e}, Long-End Loss: {le.item():.6e}, Spikiness Loss: {spl.item():.6e}", end='\r')
    return total_loss

loss = solver.step(closure)
print(f"Final training loss: {loss:.6e}")

In [7]:
# save the model
# torch.save(model.state_dict(), "deeponet_bootstrapper.pth")

# load the model
model.load_state_dict(torch.load("deeponet_bootstrapper.pth"))

<All keys matched successfully>

### Results

Qualitatively, the model tracks the market par swap curve very closely on the sample dates. The dashed model curves sit almost on top of the market markers across maturities, and the left-panel bar chart shows low residuals (basis points). With more time spend on training, this results can be improved a lot.

In [8]:
fig, dates = plot_swap_rates(model, test_market_rates_df)
fig.show()

Now we compute the Jacobian of the implied zero rates with respect to the input swap rates. We can see whether each pillar quote produces a *local and sensible* influence on the zero curve (e.g., short-tenor bumps affecting mostly the short end, longer tenors driving the long end).

In [9]:
t_grid = np.linspace(1/360, 10.0, 100)
fig = plot_jacobian(model, test_market_rates_df, t_grid=t_grid, dates=dates)
fig.show()

Finally, we plot the instantaneous forward rates for the same dates shown above. The smooth and stable forward shapes suggest that the frequency and smoothness penalties are doing their job. The shaded extrapolation region also makes explicit where the curve is driven purely by model assumptions rather than by market instruments.

One interesting feature is the hump that appears near the boundary of the information set. This hump emerges because the model must satisfy the imposed long-term forward rate: in order to anchor the long-run expectation, the curve is forced to bend upward (and then revert) close to the last liquid maturity. A large hump in this case indicates that the model is *over-correcting* in its choice of instantaneous forward curves to reconcile short-/medium-term market data with the long-term anchor. This is a warning sign that our chosen long-term level is likely inconsistent with the shapes implied by the observed curves.


In [10]:
fig = plot_instant_fwds(model, test_market_rates_df, t_max=15, dates=dates)
fig.show()

## Conclusions

Machine learning approaches to curve construction open the door to new ways of thinking about how to build yield curves. It seems that we are no longer constrained in practice to a particular set of interpolation rules; instead, we can learn features observed in the data. Still, as explained before, practitioners need to impose constraints: because we live in a discrete world of data, the selection of instantaneous forward-rate curves is an underdetermined problem. Therefore, an appropriate set of restrictions is needed to reduce the search space.